In [0]:
/* ============================================================
   Data → Decision #3 : Appointment No-Show Risk & Revenue
   Visualization Queries
   Purpose: Feed Power BI / Databricks Charts / BI Tools
   Author: Smriti Karanjit
   ============================================================ */


/* ------------------------------------------------------------
   SECTION A — Risk Band Volume & Severity
   Used for: Volume vs Severity Chart
------------------------------------------------------------ */

-- A1: Appointment Count by Risk Band
SELECT
  risk_band,
  COUNT(*) AS appointments
FROM gold.fact_noshow_scoring
GROUP BY risk_band
ORDER BY appointments DESC;


-- A2: Average No-Show Probability by Risk Band
SELECT
  risk_band,
  ROUND(AVG(no_show_probability), 3) AS avg_no_show_probability
FROM gold.fact_noshow_scoring
GROUP BY risk_band
ORDER BY avg_no_show_probability DESC;



/* ------------------------------------------------------------
   SECTION B — Financial Impact (Simulator View)
   Used for: Efficiency & Scale Charts
------------------------------------------------------------ */

-- B1: Total Expected Revenue at Risk by Band
SELECT
  risk_band,
  ROUND(SUM(expected_revenue_at_risk), 0) AS total_expected_revenue_at_risk
FROM gold.fact_noshow_scoring
GROUP BY risk_band
ORDER BY total_expected_revenue_at_risk DESC;


-- B2: Total Expected Recovered Revenue by Band
SELECT
  risk_band,
  ROUND(SUM(expected_recovered_revenue), 0) AS total_expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
GROUP BY risk_band
ORDER BY total_expected_recovered_revenue DESC;


-- B3: Expected Recovery per Outreach (Efficiency Metric)
SELECT
  risk_band,
  ROUND(SUM(expected_recovered_revenue), 0)  AS total_expected_recovered_revenue,
  COUNT(*)                                   AS appointments,
  ROUND(SUM(expected_recovered_revenue) / COUNT(*), 2) AS expected_recovery_per_contact
FROM gold.v_fact_noshow_simulator
GROUP BY risk_band
ORDER BY expected_recovery_per_contact DESC;



/* ------------------------------------------------------------
   SECTION C — Combined Decision View
   Used for: Executive Summary Table / Matrix Visual
------------------------------------------------------------ */

SELECT
  s.risk_band,
  COUNT(*) AS appointments,
  ROUND(AVG(s.no_show_probability), 3) AS avg_probability,
  ROUND(SUM(s.expected_revenue_at_risk), 0) AS total_expected_revenue_at_risk,
  ROUND(SUM(v.expected_recovered_revenue), 0) AS total_expected_recovered_revenue,
  ROUND(SUM(v.expected_recovered_revenue) / COUNT(*), 2) AS expected_recovery_per_contact
FROM gold.fact_noshow_scoring s
JOIN gold.v_fact_noshow_simulator v
  ON s.appointment_id = v.appointment_id
GROUP BY s.risk_band
ORDER BY expected_recovery_per_contact DESC;



/* ------------------------------------------------------------
   SECTION D — Geography Hotspots
   Used for: Horizontal Bar Chart
------------------------------------------------------------ */

SELECT
  neighbourhood,
  ROUND(SUM(expected_recovered_revenue), 0) AS total_expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
GROUP BY neighbourhood
ORDER BY total_expected_recovered_revenue DESC
LIMIT 20;



/* ------------------------------------------------------------
   SECTION E — Monthly Recovery Trend
   Used for: Line / Area Chart
------------------------------------------------------------ */

SELECT
  YEAR(appt_date) AS year,
  MONTH(appt_date) AS month,
  ROUND(SUM(expected_recovered_revenue), 0) AS expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
GROUP BY YEAR(appt_date), MONTH(appt_date)
ORDER BY year, month;



/* ------------------------------------------------------------
   END OF FILE
   ============================================================ */
